# Name That LLM: Qwen3-8B, exploring depth, length and the pool

**Goal:** find the settings for the final LLM run (Task 7). Notebook 10's first run
(full depth, 512 tokens, our passages only) scored 0.3641 alone on seed 7, and lifted the
blend by +0.023. Three questions remain, asked one at a time:
1. **Depth:** do the first 24 or 16 blocks (2/3 and 4/9 of the model) do as well as all
   36? They'd be faster.
2. **Length:** are 320 tokens enough? Only the longest passages are cut.
3. **The pool:** do 5,000 or 15,000 public generated passages help the LLM, as they lifted
   `lr` by +0.10 (P7 Task 3)?

If time remains, a fourth stage tries the step size (1e-4 or 2e-4) and the epochs (1 or 2).

Part of [github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm), plan P7
(Task 6).

**How we keep it honest and affordable:**
- **Two proxy folds** of seed 7 (folds 0 and 1), one per T4. A setting's score is the mean
  of the two.
- **One change per stage**, starting from the best of the stage before. Within 0.005 of the
  best, the cheaper setting wins.
- **The baseline is reused:** full depth, 512 tokens and no pool is exactly notebook 10's
  run, whose folds 0 and 1 are read from its output.
- **The pool never trains on a fold's own prompts** (`pool.pool_for_fold`). It's rebuilt
  here from the attached public dataset, with its safeguards and the 5 kept sources.
- **A time budget:** before each run, the notebook projects its minutes and skips it if it
  wouldn't finish inside Kaggle's 12-hour session. The projection starts from notebook 10's
  measured speed, and is recalibrated on each finished run.

In [ ]:
# Pinned to an exact commit, so this run is reproducible even after main moves on.
# peft (LoRA) and bitsandbytes (the 4-bit weights) aren't in Kaggle's image.
%pip install -q peft bitsandbytes "name-that-llm[track] @ git+https://github.com/Ryqn0/name-that-llm@b6799e5a2a293f264487dae2e27abdb74b7cefa2"

In [ ]:
import json
import os
import subprocess
import sys
import time
from contextlib import ExitStack
from pathlib import Path

import numpy as np
import pandas as pd
import peft
import torch
import transformers
import yaml

from name_that_llm.config import LLMConfig, load_config
from name_that_llm.cv import make_folds
from name_that_llm.data import LABELS, load_train
from name_that_llm.lm import find_model_dir
from name_that_llm.metrics import score
from name_that_llm.pool import build_pool, load_pool_sources

START = time.time()
# Kaggle's folders by default. The smoke test (tests/test_notebooks.py) runs this very code
# on a CPU, with these pointed at a tiny random Qwen, a fake pool and fake inputs.
INPUT = Path(os.environ.get("NTL_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("NTL_WORK", "/kaggle/working"))
CONFIG = os.environ.get("NTL_LLM_CONFIG", "llm")  # a config name, or a YAML path
ON_KAGGLE = "NTL_INPUT" not in os.environ
SEED, PROXY_FOLDS = 7, (0, 1)
SESSION_MIN = 11 * 60  # Kaggle stops a session at 12 hours: keep an hour's margin

try:
    import bitsandbytes

    BNB = bitsandbytes.__version__
except ImportError:
    BNB = "not installed"
print(
    f"torch {torch.__version__} (CUDA {torch.version.cuda}) | transformers "
    f"{transformers.__version__} | peft {peft.__version__} | bitsandbytes {BNB}"
)
GPUS = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", GPUS)
if ON_KAGGLE:
    assert len(GPUS) == 2, "this notebook needs the T4 x2 accelerator"

train = load_train()
y = train["model"].to_numpy()
labels = np.asarray(LABELS)
BASE = find_model_dir("8b-base", INPUT)
CFG = load_config(CONFIG, model_cls=LLMConfig).model_copy(
    update={"base_model": str(BASE), "seed": SEED}
)
folds = make_folds(train, CFG.n_splits, SEED).to_numpy()
# The depths to try: 2/3 and 4/9 of the model's blocks (24 and 16 of Qwen3-8B's 36).
TOTAL_BLOCKS = transformers.AutoConfig.from_pretrained(BASE).num_hidden_layers
DEPTHS = sorted({round(TOTAL_BLOCKS * f) for f in (2 / 3, 4 / 9)}, reverse=True)
print(CFG)
print(f"{TOTAL_BLOCKS} blocks; depths to try: {DEPTHS}")

## 1. The pool, rebuilt from the attached public dataset

The same code and safeguards as on the PC (`pool.build_pool`):
- open licenses only;
- passages flattened like ours, 30 to 320 words;
- copies of our passages and conflicting labels removed;
- each passage matched to the training prompt it's about, so no fold trains on its own
  prompts' topics;
- only the 5 sources that P7 Task 3 kept (`configs/pool.yaml`).

Counts only are printed, never a passage.

In [ ]:
pool = build_pool(load_pool_sources("pool"), INPUT, train)
POOL_PATH = WORK / "pool.parquet"
pool.to_parquet(POOL_PATH)
print(f"pool: {len(pool):,} passages, {pool['prompt_id'].notna().mean():.1%} matched to a prompt")
print(pool["model"].value_counts().describe()[["min", "max"]].to_string())

## 2. Running one setting on the two proxy folds

Each setting writes its config, then runs fold 0 on GPU 0 and fold 1 on GPU 1, as two
`python -m name_that_llm.llm` processes. If one fails, the other is stopped at once.

In [ ]:
TRAIN_PER_FOLD = int((folds != 0).sum())  # our passages a fold trains on
# Notebook 10, 36 blocks and 512 tokens: 1.017 s per training passage and epoch, 87 s to
# load the model. The cost grows with the blocks kept, less than linearly with the length
# (most passages are shorter than 320 tokens), and with the passages trained on.
SPP, LOAD_S, SCALE = 1.017, 87.0, [1.0]  # SCALE: recalibrated on every finished run
results = []


def estimate_minutes(cfg: LLMConfig) -> float:
    blocks = (cfg.n_layers or TOTAL_BLOCKS) / TOTAL_BLOCKS
    length = 0.5 + 0.5 * min(cfg.max_length, 512) / 512
    passages = TRAIN_PER_FOLD + min(cfg.pool_size, len(pool))
    seconds = LOAD_S + SPP * blocks * length * passages * cfg.epochs
    seconds += SPP * blocks * length * (folds == 0).sum() / 3  # predicting the fold
    return SCALE[0] * seconds / 60


def elapsed_minutes() -> float:
    return (time.time() - START) / 60


def on_gpu(gpu: int) -> dict:
    return {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONIOENCODING": "utf-8"}


def fold_scores(oof_rows: dict) -> dict:
    """{fold: balanced accuracy on its rows}, from {fold: probabilities}."""
    return {k: score(y[folds == k], labels[p.argmax(axis=1)]) for k, p in oof_rows.items()}


def run_setting(name: str, overrides: dict) -> dict | None:
    """Train and score one setting on the proxy folds; None if it wouldn't fit in time."""
    cfg = CFG.model_copy(update=overrides)
    minutes = estimate_minutes(cfg)
    left = SESSION_MIN - elapsed_minutes()
    print(f"--> {name}: projected {minutes:.0f} min ({left:.0f} min left)", flush=True)
    if minutes > left:
        print("    skipped: it wouldn't finish inside the session")
        return None
    folder = WORK / "runs" / name
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "llm.yaml").write_text(yaml.safe_dump(cfg.model_dump()), encoding="utf-8")
    start = time.time()
    procs = []
    with ExitStack() as logs:  # every log is closed, however the wait ends
        for gpu, k in zip((0, 1), PROXY_FOLDS, strict=True):
            cmd = [sys.executable, "-m", "name_that_llm.llm", "--config", str(folder / "llm.yaml")]
            cmd += ["--seed", str(SEED), "--folds", str(k), "--out", str(folder / f"fold{k}.npz")]
            if cfg.pool_size > 0:
                cmd += ["--pool", str(POOL_PATH)]
            log = logs.enter_context(open(folder / f"fold{k}.log", "w"))
            procs.append(
                subprocess.Popen(cmd, env=on_gpu(gpu), stdout=log, stderr=subprocess.STDOUT)
            )
        while None in (codes := [p.poll() for p in procs]):
            if any(code not in (None, 0) for code in codes):
                break  # one failed: stop the other below, rather than wait for it
            time.sleep(15)
        for p in procs:
            if p.poll() is None:
                p.kill()
                p.wait()
    failed = [k for k, p in zip(PROXY_FOLDS, procs, strict=True) if p.returncode != 0]
    if failed:
        tail = (folder / f"fold{failed[0]}.log").read_text(encoding="utf-8", errors="replace")
        print(tail[-3000:])
        raise RuntimeError(f"{name}: fold {failed[0]} failed")
    oof = {}
    for k in PROXY_FOLDS:
        with np.load(folder / f"fold{k}.npz") as npz:
            oof[k] = npz[f"fold{k}"]
    actual = (time.time() - start) / 60
    SCALE[0] *= actual / minutes  # the next projections learn from this run
    scores = fold_scores(oof)
    row = {"name": name, **overrides, **{f"fold{k}": s for k, s in scores.items()}}
    row |= {"mean": float(np.mean(list(scores.values()))), "minutes": round(actual, 1)}
    results.append(row)
    pd.DataFrame(results).to_csv(WORK / "results.csv", index=False)  # saved after every run
    print(
        f"    {name}: folds "
        + ", ".join(f"{s:.4f}" for s in scores.values())
        + f" | mean {row['mean']:.4f} | {actual:.0f} min",
        flush=True,
    )
    return row


def choose(rows: list[dict], cost) -> dict:
    """The best mean; within 0.005 of it, the cheapest setting (by `cost`)."""
    rows = [r for r in rows if r is not None]
    best = max(r["mean"] for r in rows)
    return min((r for r in rows if r["mean"] >= best - 0.005), key=cost)

## 3. The baseline: notebook 10's run, folds 0 and 1

Full depth, 512 tokens, no pool, 2 epochs at 1e-4: notebook 10 ran exactly this on the
same seed-7 folds. Its out-of-fold predictions are attached, so its folds 0 and 1 are read,
not re-run.

In [ ]:
found = sorted(INPUT.rglob(f"oof-llm-seed{SEED}.npy"))
assert len(found) == 1, f"expected notebook 10's output, found {found}"
oof10 = np.load(found[0])
base_scores = fold_scores({k: oof10[folds == k] for k in PROXY_FOLDS})
baseline = {
    "name": "baseline (notebook 10)",
    "n_layers": None,
    "max_length": 512,
    "pool_size": 0,
    **{f"fold{k}": s for k, s in base_scores.items()},
    "mean": float(np.mean(list(base_scores.values()))),
    "minutes": 0.0,
}
results.append(baseline)
print(f"baseline: mean {baseline['mean']:.4f}", base_scores)

## 4. The stages, one change at a time

Each stage starts from the best of the stage before. A stage's "cheaper" means fewer
blocks, then fewer tokens, then fewer pool passages, then fewer epochs.

In [ ]:
def cost(row: dict) -> tuple:
    return (
        row.get("n_layers") or TOTAL_BLOCKS,
        row.get("max_length", 512),
        row.get("pool_size", 0),
        row.get("epochs", 2),
    )


best = baseline
print("Stage 1: depth")
stage = [best] + [run_setting(f"layers{n}", {"n_layers": n}) for n in DEPTHS]
best = choose(stage, cost)
chosen = {"n_layers": best.get("n_layers")}
print("    chosen:", chosen)

print("Stage 2: length")
stage = [best, run_setting("len320", chosen | {"max_length": 320})]
best = choose(stage, cost)
chosen["max_length"] = best.get("max_length", 512)
print("    chosen:", chosen)

print("Stage 3: the pool")
stage = [best] + [run_setting(f"pool{n}", chosen | {"pool_size": n}) for n in (5000, 15000)]
best = choose(stage, cost)
chosen["pool_size"] = best.get("pool_size", 0)
print("    chosen:", chosen)

print("Stage 4, if time remains: the step size and the epochs")
stage = [best]
for lr, epochs in [(2e-4, 2), (1e-4, 1), (2e-4, 1)]:
    stage.append(run_setting(f"lr{lr:g}-ep{epochs}", chosen | {"lr": lr, "epochs": epochs}))
best = choose(stage, cost)
chosen |= {"lr": best.get("lr", CFG.lr), "epochs": best.get("epochs", CFG.epochs)}
print("    chosen:", chosen)

## 5. The results, and the chosen settings

Every setting that ran, and the settings Task 7's final run will use, as a YAML file
(`llm-chosen.yaml`).

In [ ]:
table = pd.DataFrame(results)
table.to_csv(WORK / "results.csv", index=False)
print(table.to_string(index=False))
final = CFG.model_copy(update={k: v for k, v in chosen.items() if v is not None or k == "n_layers"})
(WORK / "llm-chosen.yaml").write_text(yaml.safe_dump(final.model_dump()), encoding="utf-8")
print(
    "chosen:", json.dumps(chosen), f"| mean {best['mean']:.4f} | {elapsed_minutes():.0f} min in all"
)

## Findings

_Filled in after the run._